<div class="alert alert-info" style="border-radius: 10px;">
    <h1 style="text-align: center;">DATA ANALYSIS OF THE .h5 OUTPUT FILES</h1>
    <hr>
    <b>AUTHOR:</b> Luigi Costantini <br>
    <b>MAIL:</b> <a href="mailto:luigi.costantini@studenti.unipd.it">luigi.costantini@studenti.unipd.it</a>
</div>

In [ ]:
!pip install pyvirtualdisplay
%pip install tqdm
%pip install imageio[ffmpeg]

<div style="background-color: #fcfcfc; padding: 15px; border-left: 5px solid #2c3e50; border-radius: 5px; font-family: sans-serif; color: #111111;">
    <h3 style="margin-top: 0; color: #2c3e50;">📑 Table of Contents</h3>
    <p style="font-size: 14px; color: #333333;"><i>The following sections detail the analysis of the datasets generated by the CosmoLattice simulation.</i></p>
    <ul style="line-height: 1.8; color: #222222;">
        <li><b><a href="#internal_structure" style="color: #1a5276;">internal structure of HDF5 files</a></b>: internal structure of HDF5 files.</li>
        <li><b><a href="#field_3D_snapshot" style="color: #1a5276;">3D stapshot of the field</a></b>: 3D stapshot of the field, obtained from snapshot_scalar_singlet.h5 file.</li>
        <li><b><a href="#field_3D_video" style="color: #1a5276;">3D video of the field evolution</a></b>: 3D video of the field evolution, obtained from snapshot_scalar_singlet.h5 file.</li>
        <li><b><a href="#area_parameter" style="color: #1a5276;">Area parameter evolution</a></b>: Area parameter evolution, obtained from snapshot_scalar_singlet.h5 file.</li>
    </ul>
</div>

<a id="internal_structure"></a>
## Internal structure of HDF5 files

In [ ]:
import h5py

def stampa_struttura(name, obj):
    """
    Funzione di callback che viene applicata a ogni elemento (gruppo o dataset) 
    trovato all'interno del file HDF5.
    """
    indentazione = "  " * name.count('/')
    
    if isinstance(obj, h5py.Group):
        print(f"{indentazione}📁 Gruppo: {name}")
        # Stampa gli attributi del gruppo, se presenti
        for key, val in obj.attrs.items():
            print(f"{indentazione}   ↳ Attributo: {key} = {val}")
            
    elif isinstance(obj, h5py.Dataset):
        print(f"{indentazione}📊 Dataset: {name} | Shape: {obj.shape} | Tipo: {obj.dtype}")
        # Stampa gli attributi del dataset
        for key, val in obj.attrs.items():
            print(f"{indentazione}   ↳ Attributo: {key} = {val}")

def esplora_file_h5(nome_file):
    try:
        with h5py.File(nome_file, 'r') as f:
            print(f"=== Struttura del file: {nome_file} ===")
            
            # Stampa gli attributi globali (metadata salvati nella root del file)
            if f.attrs:
                print("Attributi globali (Root):")
                for key, val in f.attrs.items():
                    print(f"  {key}: {val}")
                print("-" * 40)
            
            # Esplora ricorsivamente tutto il contenuto
            f.visititems(stampa_struttura)
            
    except FileNotFoundError:
        print(f"Errore: Il file '{nome_file}' non è stato trovato nella directory corrente.")
    except Exception as e:
        print(f"Errore durante l'apertura del file: {e}")

# Esecuzione
nome_file = 'snapshot_scalar_singlet.h5'
esplora_file_h5(nome_file)

<a id="field_3D_snapshot"></a>
## 3D stapshot of the field

In [ ]:
import os
import h5py
import numpy as np
from scipy.ndimage import gaussian_filter
import pyvista as pv
import matplotlib.colors as mcolors

# --- NUOVO AVVIO DEL DISPLAY VIRTUALE ---
# Sostituisce il vecchio pv.start_xvfb() per le versioni recenti di PyVista
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1920, 1080))
display.start()
# ----------------------------------------
# =====================================================================
# 1. IMPOSTAZIONI UTENTE (Modifica questi parametri in base alle tue esigenze)
# =====================================================================

# --- Impostazioni di Salvataggio ---
bol = False
SALVA_IMMAGINE = bol                    # True = Salva uno snapshot 4K per la tesi
SALVA_VIDEO = bol                       # True = Genera un video MP4 ruotando la telecamera a 360°

# --- File e Dati ---
NOME_FILE = "snapshot_scalar_singlet.h5" # Nome del file HDF5 unico
GRUPPO_HDF5 = "S_0"                      # Nome del gruppo principale all'interno del file

# --- Scelta Temporale ---
INDICE_SCELTO = -1                      # 0 = primo istante, -1 = ultimo istante, -5 = quintultimo, ecc.

# --- Filtri Fisici ---
SIGMA_GAUSSIANO = 0.0                   # Intensità della sfocatura per lisciare le pareti topologiche
MAX_OPACITY = 0.04                      # Opacità massima da associare alle nuvole assioniche

# --- Impostazioni di Visualizzazione Spaziale (Cubo completo o sottocubo) ---
USA_SOTTOCUBO = False                   # True = Ritaglia una porzione; False = Renderizza l'intero reticolo

# Coordinate centrali e dimensione del ritaglio (ignorate se USA_SOTTOCUBO = False)
CENTRO_X, CENTRO_Y, CENTRO_Z = 100, 100, 50
OFFSET = 100                             # Il raggio del cubo (Es: 50 crea un cubo di lato 100)


# =====================================================================
# 2. RICERCA E ORDINAMENTO DEI DATASET NEL FILE HDF5
# =====================================================================
print("\n--- AVVIO RICERCA DATI ---")
print(f"Lettura della struttura del file '{NOME_FILE}'...")

available_times = []

if not os.path.exists(NOME_FILE):
    print(f"Errore: Il file {NOME_FILE} non è stato trovato nella cartella corrente.")
    exit()

try:
    with h5py.File(NOME_FILE, 'r') as f:
        if GRUPPO_HDF5 not in f:
            print(f"Errore: Il gruppo '{GRUPPO_HDF5}' non è presente nel file.")
            exit()
            
        dataset_names = list(f[GRUPPO_HDF5].keys())
        
        # Converte i nomi dei dataset (stringhe) in float per un ordinamento corretto
        for name in dataset_names:
            try:
                t_val = float(name)
                available_times.append((t_val, name))
            except ValueError:
                print(f"  [!] Avviso: Impossibile convertire in tempo il dataset '{name}'")
except Exception as e:
    print(f"Errore critico durante l'accesso preliminare al file: {e}")
    exit()

if not available_times:
    print("Errore: Nessun dataset temporale valido trovato all'interno del gruppo.")
    exit()

# Ordina tutti gli istanti di tempo trovati in ordine strettamente cronologico
available_times.sort(key=lambda x: x[0])

try:
    target_time, target_dataset = available_times[INDICE_SCELTO]
except IndexError:
    print(f"Avviso: L'indice {INDICE_SCELTO} è fuori limite. Utilizzo l'ultimo istante disponibile.")
    target_time, target_dataset = available_times[-1]

print(f"Trovati {len(available_times)} istanti temporali validi.")
print(f"Dataset selezionato: {target_dataset} (t = {target_time})")


# =====================================================================
# 3. CARICAMENTO DATI E TRATTAMENTO FISICO
# =====================================================================
print("\n--- ELABORAZIONE CAMPO FISICO ---")
print(f"Caricamento del reticolo al tempo t={target_time} in memoria...")

try:
    with h5py.File(NOME_FILE, 'r') as f:
        # Estrae i dati puntando al gruppo e poi al nome del dataset selezionato
        a_field = f[f"{GRUPPO_HDF5}/{target_dataset}"][:]
except Exception as e:
    print(f"Errore critico durante l'estrazione dei dati: {e}")
    exit()

print("Mappatura del campo assionico con sin(a)...")
mapped_field = np.sin(a_field)

print(f"Applicazione filtro Gaussiano (sigma={SIGMA_GAUSSIANO})...")
field_smoothed = gaussian_filter(mapped_field, sigma=SIGMA_GAUSSIANO)


# =====================================================================
# 4. GESTIONE SPAZIALE: RITAGLIO ED EFFETTO PAC-MAN (PBC)
# =====================================================================
print("\n--- PREPARAZIONE SPAZIALE ---")
max_x, max_y, max_z = field_smoothed.shape

if not USA_SOTTOCUBO:
    print("Modalità: CUBO COMPLETO selezionata.")
    sub_field = field_smoothed
    origine_reale = (0, 0, 0)
    titolo_plot = f"Axion Domain Walls (Full Box: {max_x}^3) | t={target_time}"
else:
    print(f"Modalità: SOTTOCUBO | Centro ({CENTRO_X}, {CENTRO_Y}, {CENTRO_Z}) | Offset {OFFSET}")
    
    # Aritmetica modulare (%) per applicare le Condizioni al Contorno Periodiche (PBC).
    idx_x = np.arange(CENTRO_X - OFFSET, CENTRO_X + OFFSET) % max_x
    idx_y = np.arange(CENTRO_Y - OFFSET, CENTRO_Y + OFFSET) % max_y
    idx_z = np.arange(CENTRO_Z - OFFSET, CENTRO_Z + OFFSET) % max_z

    origine_reale = (CENTRO_X - OFFSET, CENTRO_Y - OFFSET, CENTRO_Z - OFFSET)

    sub_field = field_smoothed[np.ix_(idx_x, idx_y, idx_z)].copy()
    titolo_plot = f"Axion Domain Walls (Cropped {OFFSET*2}^3) | t={target_time}"


# =====================================================================
# 5. CREAZIONE DELLA GRIGLIA E ISO-ESTRAZIONE MURI
# =====================================================================
print("\n--- PREPARAZIONE MOTORE GRAFICO ---")

grid = pv.ImageData(dimensions=sub_field.shape, spacing=(1, 1, 1), origin=origine_reale)
grid.point_data["AxionField"] = sub_field.flatten(order="F")

if np.min(sub_field) <= 0.0 <= np.max(sub_field):
    print("Estrazione mesh dei Muri di Dominio in corso...")
    domain_walls = grid.contour(isosurfaces=[0.0], scalars="AxionField")
    has_walls = True
else:
    has_walls = False
    print("[!] Avviso: Nel volume selezionato NON ci sono Muri di Dominio (0.0).")


# =====================================================================
# 6. DEFINIZIONE DELLA SCENA GRAFICA
# =====================================================================
def prepara_scena(p):
    # ---Aggiunta dei Muri di Dominio (Rosso) ---
    if has_walls:
        p.add_mesh(
            domain_walls,
            color='red',
            opacity=1.0,
            smooth_shading=True,
            specular=0.5           
        )
    
    # ---Aggiunta Nuvole e Onde Assioniche (Volume Rendering) ---
    cmap_custom = mcolors.LinearSegmentedColormap.from_list("AxionMap", ["blue", "black", "purple"])
    opacita = [0.0, MAX_OPACITY, 0.001, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.001, MAX_OPACITY, 0.0]

    p.add_volume(
        grid, 
        scalars="AxionField",
        cmap=cmap_custom,
        opacity=opacita,                
        clim=[-1.0, 1.0],
        shade=True,
        mapper="smart",
        show_scalar_bar=False
    )
    
    # ---Righelli con Coordinate Numeriche degli Assi ---
    p.show_grid(
        grid=False, 
        xtitle="X",
        ytitle="Y", 
        ztitle="Z",
        color="black",              
        font_size=16,               
        location="outer",           
        all_edges=True              
    )

    # --- Testo a schermo (HUD) ---
    testo_tempo = f"Time: {target_time:.4f}"
    p.add_text(
        testo_tempo, 
        position='upper_left', 
        color='black', 
        font_size=18,
        shadow=False
    )
    
    p.set_background('white')


# =====================================================================
# 7. ESPORTAZIONE AUTOMATICA IN BACKGROUND (Immagine + Video)
# =====================================================================
ANGOLO_SNAPSHOT = 180  

if SALVA_IMMAGINE or SALVA_VIDEO:
    print("\n[+] Lavoro in background: Esportazione in corso... (attendere)")
    plotter_export = pv.Plotter(off_screen=True)
    prepara_scena(plotter_export)

    if SALVA_IMMAGINE:
        posizione_iniziale_camera = plotter_export.camera.position
        plotter_export.camera.Azimuth(ANGOLO_SNAPSHOT)
        
        nome_immagine = f"Snapshot_Assioni_t{target_time:.6f}.png"
        plotter_export.screenshot(nome_immagine, window_size=[3840, 2160])
        print(f"    -> Immagine per la tesi salvata: {nome_immagine} (Angolo: {ANGOLO_SNAPSHOT}°)")
        
        plotter_export.camera.position = posizione_iniziale_camera

    if SALVA_VIDEO:
        nome_video = f"Simulazione_Assioni_t{target_time:.6f}.mp4"
        print("    -> Generazione video 3D rotante... (potrebbe volerci un minuto)")
        plotter_export.open_movie(nome_video, framerate=30)  
        
        for _ in range(180):
            plotter_export.camera.Azimuth(2) 
            plotter_export.write_frame()
        
        plotter_export.mwriter.close() 
        print(f"    -> Video salvato: {nome_video}")

    plotter_export.close() 
else:
    print("\n[+] Esportazione automatica disattivata. Passaggio diretto alla visualizzazione interattiva.")

# =====================================================================
# 8. AVVIO ESPLORAZIONE INTERATTIVA
# =====================================================================
print("\nAvvio finestra 3D interattiva... (Chiudi la finestra per terminare il programma)")

plotter_interattivo = pv.Plotter(title=titolo_plot)
prepara_scena(plotter_interattivo)

plotter_interattivo.show(jupyter_backend='static')

<a id="field_3D_video"></a>
## 3D video of the field evolution

In [ ]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import glob
import h5py
import numpy as np
import time
import subprocess
import sys
import pickle
from tqdm.auto import tqdm
import shutil
import imageio.v3 as iio
from imageio import get_writer
from IPython.display import Video, display

# =====================================================================
# 1. IMPOSTAZIONI UTENTE
# =====================================================================
DO_ZOOM = True          # True = Esegue zoom e rotazione, False = Si ferma alla fine dell'evoluzione
SALVA_ESTERNO = False   # False = Solo player su Jupyter (NO GIF, ma salva comunque un video nella cartella dei fotogrammi). 
                        # True = Salva MP4 e GIF su disco (Nella cartella principale).
NOME_VIDEO = "Simulazione_Assioni_Completa.mp4"
NOME_GIF = "Simulazione_Assioni_Completa.gif"
NOME_FILE = "snapshot_scalar_singlet.h5"
GRUPPO_HDF5 = "S_0"
CARTELLA_FRAMES = "frames_temporanei"

INDICE_INIZIALE = 0                  # Salta i primi N istanti temporali (0 = tutti, 1 = salta il primo ecc.)
FRAMERATE = 30                       # numero di fotogrammi al secondo per il video finale
FRAMES_PER_STEP = 15                 # numero di fotogrammi per ogni scena (step) presente nel file HDF5
FRAMES_ZOOM = 90                     # gestisce la fluidità dello zoom
FRAMES_ROTAZIONE = 180               # gestisce la fluidità della rotazione
FRAMES_PAUSA = int(FRAMERATE * 1.5)  # stabilisce la durata dei momenti di immobilità tra le varie fasi


SIGMA_GAUSSIANO = 0.0    #Se maggiore di 0.0, applica un gaussian_filter 3D all'intero campo prima del rendering per smussare artefatti.
MAX_OPACITY = 0.04       # Regola il picco di opacità nel volume rendering. Valori più alti rendono la nebbia scalare più solida

CENTRO_X, CENTRO_Y, CENTRO_Z = 150, 150, 150  #centro del sottocubo 
OFFSET = 50                                   #raggio del sottocubo (lato = 2*OFFSET)
num_cores = 4                                 #Determina quanti processi vengono lanciati simultaneamente per renderizzare i frame, parallelizzando il carico.

####################################################################
os.makedirs(CARTELLA_FRAMES, exist_ok=True)
TEMP_VIDEO = os.path.join(CARTELLA_FRAMES, "temp_preview.mp4")

def crea_task(frame_idx, dataset_nome, t_val, hud, usa_sottocubo, show_box, pos, focal, up, azimuth=0.0):
    return {
        'frame_id': frame_idx, 'nome_file': NOME_FILE, 'gruppo': GRUPPO_HDF5, 
        'dataset_nome': dataset_nome, 't_val': t_val, 'hud_text': hud, 
        'usa_sottocubo': usa_sottocubo, 'show_box': show_box, 
        'camera_pos': pos, 'camera_focal': focal, 'camera_up': up, 'azimuth': azimuth,
        'cartella': CARTELLA_FRAMES, 'cx': CENTRO_X, 'cy': CENTRO_Y, 'cz': CENTRO_Z, 
        'offset': OFFSET, 'sigma': SIGMA_GAUSSIANO, 'max_opacity': MAX_OPACITY
    }

# =====================================================================
# PIANIFICAZIONE
# =====================================================================
print("\n--- AVVIO RICERCA DATI ---")
available_times = []
if not os.path.exists(NOME_FILE):
    raise FileNotFoundError(f"Errore: Il file '{NOME_FILE}' non è stato trovato.")
    
with h5py.File(NOME_FILE, 'r') as f:
    if GRUPPO_HDF5 in f:
        for name in f[GRUPPO_HDF5].keys():
            try: available_times.append((float(name), name))
            except ValueError: pass

if not available_times:
    raise RuntimeError("Nessun dato temporale trovato.")

available_times.sort(key=lambda x: x[0])
files_da_processare = available_times[INDICE_INIZIALE:]

with h5py.File(NOME_FILE, 'r') as f:
    forma_reale = f[f"{GRUPPO_HDF5}/{files_da_processare[0][1]}"].shape
    
centro_reale_z, centro_reale_y, centro_reale_x = forma_reale[0]/2.0, forma_reale[1]/2.0, forma_reale[2]/2.0
start_focal = np.array([centro_reale_x, centro_reale_y, centro_reale_z])
end_focal = np.array([CENTRO_X, CENTRO_Y, CENTRO_Z])
dir_v = np.array([1.0, 1.0, 0.8]) 
dir_norm = dir_v / np.linalg.norm(dir_v)
start_dist = max(forma_reale) * 2.8 
end_dist = (OFFSET * 2) * 2.8   
start_pos = start_focal + (dir_norm * start_dist)
end_pos = end_focal + (dir_norm * end_dist)
start_up = np.array([0.0, 0.0, 1.0])

tasks_to_render = []
tasks_to_clone = []
frame_idx = 0
ultimo_t, ultimo_dataset = files_da_processare[-1]

# Fase 1: Globale
for t_val, dataset_nome in files_da_processare:
    hud = f"Time: {t_val:.4f}"
    base_idx = frame_idx 
    for j in range(FRAMES_PER_STEP):
        if j == 0: tasks_to_render.append(crea_task(frame_idx, dataset_nome, t_val, hud, False, False, start_pos, start_focal, start_up))
        else: tasks_to_clone.append((base_idx, frame_idx))
        frame_idx += 1

if DO_ZOOM:
    # Fase 2: Zoom
    for i in range(FRAMES_ZOOM):
        alpha_lineare = i / float(FRAMES_ZOOM - 1)
        alpha_smooth = alpha_lineare * alpha_lineare * (3 - 2 * alpha_lineare)
        c_pos = (1 - alpha_smooth) * start_pos + alpha_smooth * end_pos
        c_foc = (1 - alpha_smooth) * start_focal + alpha_smooth * end_focal
        hud = f"Time: {ultimo_t:.4f}"
        tasks_to_render.append(crea_task(frame_idx, ultimo_dataset, ultimo_t, hud, False, False, c_pos, c_foc, start_up))
        frame_idx += 1

    # Fase 2.5: Transizioni
    hud_25 = f"Time: {ultimo_t:.4f}"
    for state_sottocubo, state_box in [(False, False), (False, True), (True, True)]:
        base_idx = frame_idx
        for j in range(FRAMES_PAUSA): 
            if j == 0: tasks_to_render.append(crea_task(frame_idx, ultimo_dataset, ultimo_t, hud_25, state_sottocubo, state_box, end_pos, end_focal, start_up))
            else: tasks_to_clone.append((base_idx, frame_idx))
            frame_idx += 1

    # Fase 3: Rotazione
    gradi_frame = 360.0 / FRAMES_ROTAZIONE
    for i in range(FRAMES_ROTAZIONE):
        tasks_to_render.append(crea_task(frame_idx, ultimo_dataset, ultimo_t, hud_25, True, False, end_pos, end_focal, start_up, azimuth=i*gradi_frame))
        frame_idx += 1

# Salvataggio fisico dei Task per i worker
with open('tasks_to_render.pkl', 'wb') as f:
    pickle.dump(tasks_to_render, f)

# =====================================================================
# RENDERING FARM LOCALE TRAMITE SUBPROCESS
# =====================================================================
print(f"\n--- AVVIO RENDERING SU {num_cores} CORE ---")
active_procs = []
task_idx = 0
pbar = tqdm(total=len(tasks_to_render), desc="Rendering")

while task_idx < len(tasks_to_render) or active_procs:
    for p in active_procs[:]:
        if p.poll() is not None:
            active_procs.remove(p)
            pbar.update(1)
            
    while len(active_procs) < num_cores and task_idx < len(tasks_to_render):
        # QUI E' AVVENUTA LA MODIFICA AL NOME DEL FILE
        frame_id = tasks_to_render[task_idx]['frame_id']
        expected_frame_path = os.path.join(CARTELLA_FRAMES, f"frame_{frame_id:05d}.png")
        if os.path.exists(expected_frame_path):
            task_idx += 1
            pbar.update(1)
            continue
        cmd = [sys.executable, "_funzioni_video.py", "tasks_to_render.pkl", str(task_idx)]
        proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        active_procs.append(proc)
        task_idx += 1
        
    time.sleep(0.05) 
    
pbar.close()

# --- CLONAZIONE FISICA DEI FRAME ---
print(f"\n--- CLONAZIONE FRAME STATIC DI PAUSA ({len(tasks_to_clone)} frame) ---")
for source_id, target_id in tqdm(tasks_to_clone, desc="Clonazione"):
    source_path = os.path.join(CARTELLA_FRAMES, f"frame_{source_id:05d}.png")
    target_path = os.path.join(CARTELLA_FRAMES, f"frame_{target_id:05d}.png")
    if not os.path.exists(target_path) and os.path.exists(source_path):
        shutil.copy(source_path, target_path)

# =====================================================================
# MONTAGGIO VIDEO E MOSTRA SU JUPYTER
# =====================================================================
print(f"\n--- MONTAGGIO IN CORSO ---")
frame_files = sorted(glob.glob(os.path.join(CARTELLA_FRAMES, "frame_*.png")))

if frame_files:
    video_target = NOME_VIDEO if SALVA_ESTERNO else TEMP_VIDEO
    
    writer_mp4 = get_writer(video_target, fps=FRAMERATE, format='FFMPEG', codec='libx264', pixelformat='yuv420p')
    writer_gif = get_writer(NOME_GIF, fps=FRAMERATE, format='GIF', loop=0) if SALVA_ESTERNO else None
    
    desc_msg = "Creazione MP4 + GIF" if SALVA_ESTERNO else "Creazione MP4 Temporaneo"
    for f in tqdm(frame_files, desc=desc_msg):
        img = iio.imread(f)
        writer_mp4.append_data(img)
        if writer_gif:
            writer_gif.append_data(img)
            
    writer_mp4.close()
    if writer_gif:
        writer_gif.close()
        
    print(f"\n[+] RENDERING E MONTAGGIO COMPLETATI!")
    if SALVA_ESTERNO:
        print(f"[+] Salvati su disco: {NOME_VIDEO} e {NOME_GIF}")
    else:
        print(f"[+] GIF ignorata. Video generato in modalità anteprima locale.")
    
    # ---> VISUALIZZAZIONE DEL VIDEO IN JUPYTER <---
    print(f"\n[>] Avvio del player video...")
    display(Video(video_target, embed=True, width=800, html_attributes="controls loop autoplay"))
    
else:
    print("\n[!] Nessun frame trovato per il montaggio.")


<a id="area_parameter"></a>
## Area parameter evolution

NOTA: Questo blocco genera file di testo all'interno della cartella "analisi_cumulativa". Tali file, provenienti da più simulazioni differenti, vengono usati assieme nel blocco "area parameter evolution" del file "Analisi_Cumulativa.jpynb" per calcolare media e deviazione standard.

In [ ]:
import h5py                  
import numpy as np           
import matplotlib.pyplot as plt 
import os                    
from datetime import datetime 
from skimage import measure   
from scipy.interpolate import interp1d

# ==========================================
# 1. PARAMETRI DI CONFIGURAZIONE
# ==========================================
NOME_FILE_H5 = "snapshot_scalar_singlet.h5"
GRUPPO_HDF5 = "S_0"
NOME_FILE_TXT = "average_scale_factor.txt"

# Frazione del reticolo da analizzare (1 = 100%, 0.5 = 50% centrale).
fraction_subcube = 1.0

# Tempo di partenza dell'analisi
t_start = 0.0  

# --- CONTROLLI DI SALVATAGGIO ---
SALVA_DATI = True      # Se True, salva i risultati in un file .txt
SALVA_IMMAGINE = False  # Se True, salva il grafico in un file .png ad alta risoluzione

# --- GESTIONE CARTELLA DI OUTPUT ---
cartella_output = "analisi_cumulativa"      # cartella in cui saranno salvati gli eventuali file di output
os.makedirs(cartella_output, exist_ok=True) # Crea la cartella se non esiste

current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
# Indirizza i file nella nuova sottocartella
output_filename = os.path.join(cartella_output, f"Area_Parameter_{current_time}.txt")
image_filename = os.path.join(cartella_output, f"Area_Parameter_Plot_{current_time}.png")

# ==========================================
# 2. LETTURA DEL FATTORE DI SCALA
# ==========================================
print(f"Lettura del fattore di scala da '{NOME_FILE_TXT}'...")
if not os.path.exists(NOME_FILE_TXT):
    raise FileNotFoundError(f"Errore: File '{NOME_FILE_TXT}' non trovato nella directory corrente.")

# np.loadtxt gestisce automaticamente i file a colonne.
# data_txt[:, 0] = tempo, data_txt[:, 1] = a(t)
data_txt = np.loadtxt(NOME_FILE_TXT)
t_txt = data_txt[:, 0]
a_txt = data_txt[:, 1]

# Creiamo l'interpolatore per ricavare a(t) per qualsiasi t esatto dell'HDF5
get_scale_factor = interp1d(t_txt, a_txt, kind='linear', fill_value="extrapolate")

# ==========================================
# 3. RICERCA DATI NEL FILE HDF5
# ==========================================
print(f"Analisi del file '{NOME_FILE_H5}'...")
if not os.path.exists(NOME_FILE_H5):
    raise FileNotFoundError(f"Errore: File '{NOME_FILE_H5}' non trovato.")

available_times = []
with h5py.File(NOME_FILE_H5, 'r') as f:
    if GRUPPO_HDF5 not in f:
        raise KeyError(f"Errore: Gruppo '{GRUPPO_HDF5}' non trovato nel file HDF5.")
        
    for name in f[GRUPPO_HDF5].keys():
        try:
            t_val = float(name)
            if t_val >= t_start:
                available_times.append((t_val, name))
        except ValueError:
            pass

if not available_times:
    raise RuntimeError(f"Nessun dato valido trovato nel gruppo {GRUPPO_HDF5} a partire da t = {t_start}.")

available_times.sort(key=lambda x: x[0])
print(f"=> Trovati {len(available_times)} snapshot validi a partire da t = {t_start}.")

# ==========================================
# 4. CALCOLO PARAMETRO DI AREA
# ==========================================
t_values = []
Av_values = []

print(f"Inizio calcolo Marching Cubes (Sotto-cubo: {fraction_subcube*100}%)...")

with h5py.File(NOME_FILE_H5, 'r') as f:
    for t_val, dataset_name in available_times:
        print(f"Elaborazione tempo t = {t_val:.3f} ...", end="\r")
        
        try:
            a_field = f[f"{GRUPPO_HDF5}/{dataset_name}"][:]
            scale_factor = get_scale_factor(t_val)
            
            N = a_field.shape[0]
            dx = 1.0 / N 
            
            N_sub = int(N * fraction_subcube)
            start_idx = (N - N_sub) // 2
            end_idx = start_idx + N_sub
            
            L_sub = N_sub * dx
            V_comovente_sub = L_sub**3

            a_field_sub = a_field[start_idx:end_idx, start_idx:end_idx, start_idx:end_idx]

            # Mappatura del campo In-place
            np.sin(a_field_sub, out=a_field_sub)

            Aw_sub = 0.0
            
            try:
                verts, faces, normals, values = measure.marching_cubes(
                    a_field_sub, 
                    level=0.0, 
                    spacing=(dx, dx, dx)
                )
                Aw_sub = measure.mesh_surface_area(verts, faces)
            except ValueError:
                # Nessuna superficie (isocontorno) trovata
                pass

            if Aw_sub > 0:
                Av = (Aw_sub * t_val) / (scale_factor * V_comovente_sub)
            else:
                Av = 0.0
            
            t_values.append(t_val)
            Av_values.append(Av)
            
        except Exception as e:
            print(f"\nErrore al tempo t = {t_val}: {e}")

print("\nCalcolo completato!                                      ")

# ==========================================
# 5. SALVATAGGIO DATI E GRAFICO
# ==========================================
if SALVA_DATI:
    print(f"Salvataggio risultati in '{output_filename}'...")
    try:
        with open(output_filename, 'w') as txt_file:
            txt_file.write("# Time(t)\tArea_Parameter(Av)\n")
            for t, av in zip(t_values, Av_values):
                txt_file.write(f"{t:.6f}\t{av:.6f}\n")
        print("Salvataggio dati completato con successo!")
    except Exception as e:
        print(f"Errore nel salvataggio del file txt: {e}")
else:
    print("Salvataggio dati disabilitato.")

plt.figure(figsize=(10, 6))
plt.plot(t_values, Av_values, marker='o', linestyle='-', color='crimson', linewidth=2, markersize=5, label=r'$\mathcal{A}_v(t)$')
plt.title(f"Axion Domain Wall Area Parameter\nSub-cube: {fraction_subcube*100}%, Start time: {t_start}", fontsize=15, fontweight='bold')
plt.xlabel("Time (program units)", fontsize=13)
plt.ylabel(r"Area Parameter $\mathcal{A}_v$", fontsize=13)
plt.ylim(0, 1.25)

if t_values:
    plt.xlim(t_start, max(t_values))
else:
    plt.xlim(t_start, 50)

plt.grid(True, which="major", ls="--", alpha=0.6)
plt.grid(True, which="minor", ls=":", alpha=0.3)
plt.minorticks_on()
plt.legend(fontsize=12)
plt.tight_layout()

if SALVA_IMMAGINE:
    print(f"Salvataggio immagine in '{image_filename}'...")
    try:
        plt.savefig(image_filename, dpi=300, bbox_inches='tight')
        print("Salvataggio immagine completato con successo!")
    except Exception as e:
        print(f"Errore nel salvataggio dell'immagine: {e}")
else:
    print("Salvataggio immagine disabilitato.")

plt.show()